In [24]:
# mlflow for MLOPS
import os
import mlflow
import xgboost as xgb
from pathlib import Path
import pandas as pd
import logging
from sklearn.metrics import roc_auc_score,average_precision_score,precision_recall_curve,roc_curve,auc
import matplotlib.pyplot as plt
import numpy as np
from mlflow.models.signature import infer_signature
#use logging

In [2]:
# Configuration
EXPERIMENT_NAME = "Credit_default_XGBoost_Prod"
DATA_PATH = Path("../data/processed")
MODEL_OUTPUT = Path("models")
MODEL_OUTPUT.mkdir(exist_ok=True)

#xgb tuned parameters
XGB_PARAMS = {
    "n_estimators":500, 
    "learning_rate":0.08, 
    "scale_pos_weight":3.52,#23364/6636
    "tree_method":'exact', 
    "n_jobs":1, 
    "random_state":42
}

In [ ]:
# Read Data using Data Loader
# function to read the data

# read train, val test csv

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def data_loader(data_path):
    datasets ={}

    for split in ['train','val','test']:
        file_path = data_path/f"{split}.csv"

        if not file_path.exists():
            raise FileNotFoundError("Missing Expected File: {file_path} ")
        
        df = pd.read_csv(file_path)
        logging.info(f"Loaded {split}.csv with {df.shape[0]} rows and  {df.shape[1]} columns")

        # data quality checks: columns: check for column consistency against the training set
        if split == 'train':
            expected_columns = df.columns.to_list()
        else:
            if not df.columns.equals(pd.Index(expected_columns)):
                missing = set(expected_columns)- set(df.columns)
                extra = set(df.columns) - set(expected_columns)
                raise ValueError(f"Column mismatch in {split}.csv! \nMising: {missing} \nExtra: {extra}")

        datasets[split] =df

    return datasets['train'],datasets['val'],datasets['test']


try:
    train, val, test = data_loader(DATA_PATH)
except Exception as e:
    logging.error(f"Data Loading Failed: {e}")


INFO: Loaded train.csv with 24000 rows and  24000 columns
INFO: Loaded val.csv with 4800 rows and  4800 columns
INFO: Loaded test.csv with 1200 rows and  1200 columns


In [ ]:
# feature engineering
# atomic-> easy to modify features
# To do : unit tests
# .copy to prevent leakages
# To do: create module
def extract_features(raw_data):
    raw_data = raw_data.copy()

    # 1. Payment-to-Bill Ratios 
    # We use np.where to handle the -1 case specifically
    denom_1 = raw_data['BILL_AMT1'] + 1
    raw_data['pay_bill_ratio_1'] = np.where(denom_1 != 0, raw_data['PAY_AMT1'] / denom_1, 0)
    
    # For the average, we calculate the denominator first
    avg_bill_denom = raw_data[['BILL_AMT1','BILL_AMT2','BILL_AMT3']].mean(axis=1)
    # If the average bill is 0, the ratio is 0. No more .replace(0, 1) needed.
    raw_data['pay_bill_ratio_avg'] = np.where(avg_bill_denom != 0, 
                                             raw_data[['PAY_AMT1','PAY_AMT2','PAY_AMT3']].mean(axis=1) / avg_bill_denom, 
                                             0)

    # 2. Payment Delay Score 
    delay_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
    raw_data['total_delays'] = raw_data[delay_cols].clip(lower=0).sum(axis=1)
    raw_data['recent_delays'] = raw_data[['PAY_0','PAY_2']].clip(lower=0).sum(axis=1)

    #  3. Credit Utilization
    denom_util = raw_data['LIMIT_BAL'] + 1
    raw_data['utilization'] = np.where(denom_util != 0, raw_data['BILL_AMT1'] / denom_util, 0)

    # 4. Payment Momentum 
    raw_data['pay_trend'] = raw_data['PAY_AMT6'] - raw_data['PAY_AMT1'] 

    # 5. Demographic Risk 
    raw_data['age_sex_risk'] = raw_data['AGE'] * (raw_data['SEX'] == 2) 

    return raw_data

In [ ]:
def prepare_model_data(df, target_col='default', drop_cols = ['ID']):
    
    #extract features
    df_processed = extract_features(df)

    # Separate target
    y = df_processed[target_col].copy()

    # drop unwanted columns, ID
    X = df_processed.drop(columns=[target_col] + drop_cols, errors ='ignore')

    return X,y

In [ ]:
X_train, y_train = prepare_model_data(train)
X_val, y_val = prepare_model_data(val)
X_test, y_test = prepare_model_data(test)


# Final validation before training starts: # preflight check
assert not np.isinf(X_train).values.any(), "Inf detected in features!"
assert not X_train.isnull().values.any(), "NaN detected in features!"
print("Data is numerically stable. Ready for XGBoost.")

Data is numerically stable. Ready for XGBoost.


In [26]:
#get aboslute path to project folder
tracking_uri = "file://"+ os.path.abspath("mlruns")

mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow is tracking at: {tracking_uri}")

MLflow is tracking at: file:///Users/nicholus/Documents/GitHub/credit-card-default/notebooks/mlruns


In [23]:
# wrap training in mlflow training wrapper
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="XGB_Production_v1"):

    # Log Parameters
    mlflow.log_params(XGB_PARAMS)

    #Initialise the model with early stopping
    model = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=50)

    #Train

    model.fit(
        X_train, y_train,
        eval_set = [(X_val, y_val)],
        verbose=False
    )

    X_sample = X_train.astype(float) # to avoid integer warning at inference

    # create signature
    signature = infer_signature(X_sample,model.predict(X_sample))

    # Create an input sample
    input_sample = X_sample.iloc[[0]]

    # Save the model and the signature
    mlflow.xgboost.log_model(
        xgb_model=model,
        name="model",
        signature=signature,
        input_example=input_sample,
        registered_model_name="CreditDefault_XGB"
    )
    
    # Log validation metrics
    val_auc = model.best_score
    mlflow.log_metric("val_auc", val_auc)

    #Log the model artifacts
    #mlflow.sklearn.log_model(model,name="model") =< double logging

    #Save the column list
    feature_list = X_train.columns.tolist()
    mlflow.log_dict(feature_list,"feature_list.json")

      # get prediction probabilities
    test_probs = model.predict_proba(X_test)[:,1]

    # calculate the standard ROC-AUC
    test_roc_auc = roc_auc_score (y_test, test_probs)
    mlflow.log_metric("test_roc_auc",test_roc_auc)

    # calculate the PR-AUC (Average Precision)
    test_pr_auc = average_precision_score(y_test,test_probs)
    mlflow.log_metric("test_pr_auc",test_pr_auc)

    print(f"ROC-AUC:{test_roc_auc:.4f}")
    print(f"PR-AUC:{test_pr_auc:.4f}")

    # Save the PR curve plot as an  MLflow Artifact
    precision, recall, _ = precision_recall_curve(y_test, test_probs)

    plt.figure(figsize=(8,6))
    plt.plot(recall, precision, label=f"PR Curve (AUC={test_pr_auc: .2f})")
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.savefig("pr_curve.png")
    mlflow.log_artifact("pr_curve.png")
    plt.close()


    # calculate the FPR, TPR, and Thresholds
    fpr, tpr, thresholds = roc_curve(y_test,test_probs)
    # test_roc_auc already logged
    # create chart
    plt.figure (figsize=(8,6))
    plt.plot(fpr,tpr,color='darkorange',lw=2, label=f"AUC ={test_roc_auc:.2f}")
    plt.plot([0,1],[0,1], color='navy',lw=2,linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver Operating Characteristic (ROC) Curve")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.savefig("roc_curve.png")
    mlflow.log_artifact("roc_curve.png")
    plt.close()

    #log the thresholds
    thresh_df = pd.DataFrame({'threshold': thresholds,'fpr':fpr, 'tpr':tpr})
    thresh_df.to_csv("roc_threshold_data.csv", index=False)
    mlflow.log_artifact("roc_threshold_data.csv")


print(f"production run complete")


/Users/nicholus/Documents/GitHub/credit-card-default/card_default/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [14:29:41] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


ROC-AUC:0.7642
PR-AUC:0.5388
production run complete


Registered model 'CreditDefault_XGB' already exists. Creating a new version of this model...
Created version '3' of model 'CreditDefault_XGB'.


In [16]:
X_train.iloc[[0]]

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT4,PAY_AMT5,PAY_AMT6,pay_bill_ratio_1,pay_bill_ratio_avg,total_delays,recent_delays,utilization,pay_trend,age_sex_risk
0,160000,2,2,2,33,2,2,3,2,0,...,6100,12300,6100,0.092723,0.029824,9,4,1.011062,-8900,33
